<a href="https://colab.research.google.com/github/Roopanshi-Marwaha/Fraud_Ring_Detection/blob/main/Fraud_Ring_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()
#as i have them locally so just picking from there using this

Saving accounts.csv to accounts (1).csv
Saving alerts.csv to alerts (1).csv
Saving transactions.csv to transactions (1).csv


In [2]:
import pandas as pd

accounts = pd.read_csv('accounts.csv')
transactions = pd.read_csv('transactions.csv')
alerts = pd.read_csv('alerts.csv')

print(accounts.shape, transactions.shape, alerts.shape)
#printed rows and columns-->shape

(10000, 7) (1323234, 8) (1719, 9)


ACCOUNTS

In [3]:
accounts.head()
#to see first 5 rows
#TX_BEHAVIOR_ID--> a group number controlling how that account normally transacts (used to simulate realistic behavior)
# IS_FRAUD here means the account itself is a known fraud account, not just one transaction

,ACCOUNT_ID,CUSTOMER_ID,INIT_BALANCE,COUNTRY,ACCOUNT_TYPE,IS_FRAUD,TX_BEHAVIOR_ID
0,0,C_0,184.44,US,I,False,1
1,1,C_1,175.80,US,I,False,1
2,2,C_2,142.06,US,I,False,1
3,3,C_3,125.89,US,I,False,1
4,4,C_4,151.13,US,I,False,1


In [4]:
accounts['ACCOUNT_TYPE'].value_counts()
#ACCOUNT_TYPE: all 10,000 accounts are "I" so there's only one account type in this dataset, not a mix.
# That column won't be useful for us later (no variation = no signal)

,count
ACCOUNT_TYPE,
I,10000


In [5]:
accounts['IS_FRAUD'].value_counts()
# IS_FRAUD: 1,685 out of 10,000 accounts (~16.8%) are fraud accounts. That's a high fraud rate so plenty of fraud examples to learn from, not a rare event problem

,count
IS_FRAUD,
False,8315
True,1685


In [6]:
accounts['COUNTRY'].value_counts()
#all US so this column is also useless, no variation.

,count
COUNTRY,
US,10000


In [7]:
accounts['TX_BEHAVIOR_ID'].value_counts()
#exactly 5 behavior groups, 2,000 accounts each, perfectly even split

,count
TX_BEHAVIOR_ID,
1,2000
2,2000
3,2000
4,2000
5,2000


In [8]:
pd.crosstab(accounts['TX_BEHAVIOR_ID'], accounts['IS_FRAUD'])
#This will show fraud count per behavior group

#fraud rates:-
# Group 1: 291/2000 = 14.6%
# Group 2: 330/2000 = 16.5%
# Group 3: 309/2000 = 15.5%
# Group 4: 358/2000 = 17.9%
# Group 5: 397/2000 = 19.9%
# There's a mild upward trend (group 5 has more fraud than group 1) but it's not a huge gap.

IS_FRAUD,False,True
TX_BEHAVIOR_ID,,
1,1709,291
2,1670,330
3,1691,309
4,1642,358
5,1603,397


So in accounts.csv. We now know:

10k accounts are there

only useful columns are IS_FRAUD (target-ish) and TX_BEHAVIOR_ID (weak feature),

COUNTRY & ACCOUNT_TYPE are dead weight.



TRANSACTIONS

In [11]:
transactions.info()
transactions.head()

#things cleared:-
# ALERT_ID = -1 matlab is transaction ka koi alert nahi (normal transaction)
# TIMESTAMP numbers mein hai (0, 1, 2...) — real dates nahi, simulation ke "time steps" hain

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1323234 entries, 0 to 1323233
Data columns (total 8 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   TX_ID                1323234 non-null  int64  
 1   SENDER_ACCOUNT_ID    1323234 non-null  int64  
 2   RECEIVER_ACCOUNT_ID  1323234 non-null  int64  
 3   TX_TYPE              1323234 non-null  object 
 4   TX_AMOUNT            1323234 non-null  float64
 5   TIMESTAMP            1323234 non-null  int64  
 6   IS_FRAUD             1323234 non-null  bool   
 7   ALERT_ID             1323234 non-null  int64  
dtypes: bool(1), float64(1), int64(5), object(1)
memory usage: 71.9+ MB


,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID
0,1,6456,9069,TRANSFER,465.05,0,False,-1
1,2,7516,9543,TRANSFER,564.64,0,False,-1
2,3,2445,9356,TRANSFER,598.94,0,False,-1
3,4,2576,4617,TRANSFER,466.07,0,False,-1
4,5,3524,1773,TRANSFER,405.63,0,False,-1


In [13]:
transactions['TX_TYPE'].value_counts()
#TX_TYPE — sab kuch "TRANSFER" hi hai, koi variation nahi. Yeh column bhi useless nikla hai

,count
TX_TYPE,
TRANSFER,1323234


In [14]:
transactions['IS_FRAUD'].value_counts()
#IS_FRAUD — sirf 1,719 fraud transactions out of 13,23,234 = 0.13% -->bahut zyada imbalanced data
# so agar directly model train karenge toh woh "sab kuch not fraud hai" bol ke bhi 99.87% accuracy de dega but will be useless.

#now observation:-
# Account level pe fraud rate tha 16.8% (accounts.csv)
# Transaction level pe fraud rate hai sirf 0.13% (transactions.csv)
#so this means:-
# ek fraud account bhi apni zyadatar transactions normal karta hai, sirf kuch hi transactions actually "fraud-flagged" hoti hain us account ki.
# Yeh real duniya jaisa hi hai, aise account bhi mostly normal looking transfers karte hai, sirf kabhi kabhi suspicious wala move hota hai.

,count
IS_FRAUD,
False,1321515
True,1719


In [16]:
print(transactions['TIMESTAMP'].min())
print(transactions['TIMESTAMP'].max())
print(transactions['TIMESTAMP'].nunique())

#Toh simulation mein 200 discrete time units the, aur 1.3M transactions un 200 steps mein spread hain.
#kitni jaldi jaldi paisa move ho raha hai--> yeh voh bata sakta hai

0
199
200


In [17]:
transactions['TX_AMOUNT'].describe()
#Max = ₹2.14 crore — ek bahut hi bada outlier
# mean median se bohot zyada hai iska matlab hai distribution is right skewed.
#yani zyadatar transactions chhoti hain, lekin kuch bahut hi bade outliers hain jo average ko upar kheech rahe hain.

,TX_AMOUNT
count,1.323234e+06
mean,1.159882e+05
std,1.320091e+06
min,0.000000e+00
25%,2.393000e+01
50%,1.567100e+02
75%,4.400000e+02
max,2.147484e+07


In [18]:
transactions.groupby('IS_FRAUD')['TX_AMOUNT'].describe()

#Fraud transactions: amount sirf ₹2.54 se ₹19.92 tak --> bohot chhote amounts hain (max ₹20!)
#Non fraud transactions: amount ₹0 se ₹2.14 crore tak --> bahut bada range

#Matlab very intresting pattern seen:-
#jitna socha tha uska ulta hai
# log sochte hain fraud = bada amount,
# lekin yahan fraud transactions chhote chhote hain.
# Yeh actually real world money laundering pattern se match karta hai: "structuring" ya "smurfing"
# i.e bade amount ko jaanbujh ke chhote chhote pieces mein tod dena taaki bank ke threshold based alerts (jo bade amount pe trigger hote hain) trigger na ho.

,count,mean,std,min,25%,50%,75%,max
IS_FRAUD,,,,,,,,
False,1321515.0,116139.044522,1.320942e+06,0.00,24.29,156.97,440.05,21474836.47
True,1719.0,9.763310,5.928078e+00,2.54,3.78,10.60,15.30,19.92


ALERTS

In [19]:
alerts['ALERT_TYPE'].value_counts()

# cycle (936 cases): A→B→C→A. Paisa ek chain mein ghoom ke wapas apne origin ke paas ya kisi related account tak aata hai.
# fan_in (783 cases): bahut saare alag accounts se ek single account mein paisa aana.
# example, 20 chhote mule accounts, sab ek hi "collector" account ko chhoti chhoti amounts bhejte hain,
# jo baad mein woh sara paisa ek saath nikaal leta hai ya aage forward karta hai.

,count
ALERT_TYPE,
cycle,936
fan_in,783


In [20]:
alerts.head()

# ALERT_ID = 377 do baar aaya hai (row 1 aur row 3), same TX_AMOUNT (10.27) ke saath lekin alag TX_ID, alag sender/receiver, alag timestamp.
# Iska matlab: ek ALERT_ID = ek poora fraud ring, aur uss ring ke multiple transactions hote hain jo saath mein table mein listed hain. Matlab agar cycle hai A→B→C→A, toh teeno transactions (A→B, B→C, C→A) same ALERT_ID share karenge, alag alag rows mein

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
0,193,fan_in,True,82,6976,9739,TRANSFER,4.85,0
1,377,cycle,True,949,5776,2570,TRANSFER,10.27,0
2,189,fan_in,True,6280,9999,9530,TRANSFER,2.74,1
3,377,cycle,True,7999,1089,7352,TRANSFER,10.27,1
4,130,fan_in,True,12975,7025,9708,TRANSFER,3.53,2


In [21]:
print(alerts['ALERT_ID'].nunique())
print(len(alerts))
print(alerts.groupby('ALERT_ID').size().describe())


# 391 unique fraud rings total (cycle + fan_in dono milake)
# Har ring mein average ~4.4 transactions hote hain
# Sabse chhota ring = 1 transaction (WEIRD AS "ring" mein toh kam se kam 2-3 transactions honi chahiye, single transaction wala "ring" nahi ho sakta typically)--->OUTLIER
# Sabse bada ring = 5 transactions, aur woh bhi bahut consistent hai (75th percentile bhi 5 hai) so matlab zyadatar rings ka size 4 ya 5 hi hai, bahut tight range, Yeh bahut structured/simulated pattern hai real life mei not that consistent but stimulation ke liye works

391
1719
count    391.000000
mean       4.396419
std        0.678526
min        1.000000
25%        4.000000
50%        4.000000
75%        5.000000
max        5.000000
dtype: float64


In [22]:
ring_sizes = alerts.groupby('ALERT_ID').size()
single_tx_rings = ring_sizes[ring_sizes == 1]
print(single_tx_rings)

ALERT_ID
44     1
231    1
272    1
dtype: int64


In [23]:
alerts[alerts['ALERT_ID'].isin([44, 231, 272])]

# Ye 3 single-tx rings (231, 272, 44) sab timestamp 198/199 pe hain
# simulation ke last steps. Ring poora complete hone se pehle hi simulation khatam ho gayi (200 steps ka limit), isliye baaki  transactions record nahi hue.
# Data error nahi, bas edge effect hai jo kisi bhi time bounded simulation mein ho sakta hai (jaise ek movie ka last scene achanak kat jaye).
# Feature engineering mein to remember: kuch rings incomplete honge sirf time cutoff ki wajah se.

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
1704,231,cycle,True,1312016,3040,1565,TRANSFER,12.14,198
1714,272,cycle,True,1316271,2465,707,TRANSFER,16.31,198
1715,44,fan_in,True,1316636,1453,8709,TRANSFER,2.81,199
